In [ ]:
import re
import pandas as pd
import requests
from tqdm.notebook import tqdm

In [ ]:
df = pd.read_csv('data/hexamagdb.csv', low_memory=False)
num = ['a_A', 'c_A', 'V_A3', 'Ms_emu_g', 'Mr_emu_g', 'Hc_kOe', 'Mr_Ms', 'Ha_kOe', 'K_erg_cm3', 'K_erg_g', 'Tc_K']
df[num] = df[num].apply(pd.to_numeric, errors='coerce')

In [ ]:
#checking lattice parameters outside 5.7-6.1 A for a and 22.5-23.6 A for c
df[(df['a_A'] < 5.7) | (df['a_A'] > 6.1) | (df['c_A'] < 22.5) | (df['c_A'] > 23.6)][['Record_ID', 'Formula', 'a_A', 'c_A', 'DOI']]

In [ ]:
#checking that the unit-cell volume agrees with a and c within 1%
V_calc = 3**0.5 / 2 * df['a_A']**2 * df['c_A']
df[(df['V_A3'] - V_calc).abs() / V_calc > 0.01][['Record_ID', 'Formula', 'a_A', 'c_A', 'V_A3', 'DOI']]

In [ ]:
#checking that Mr does not exceed Ms
df[df['Mr_emu_g'] > df['Ms_emu_g']][['Record_ID', 'Formula', 'Ms_emu_g', 'Mr_emu_g', 'DOI']]

In [ ]:
#checking that the squareness ratio agrees with Mr/Ms
df[(df['Mr_Ms'] - df['Mr_emu_g'] / df['Ms_emu_g']).abs() > 0.02][['Record_ID', 'Formula', 'Ms_emu_g', 'Mr_emu_g', 'Mr_Ms', 'DOI']]

In [ ]:
#checking Hc above 10 kOe or below 50 Oe and Tc outside 550-800 K (units, Hc vs Ha)
df[(df['Hc_kOe'] > 10) | (df['Hc_kOe'] < 0.05) | (df['Tc_K'] < 550) | (df['Tc_K'] > 800)][['Record_ID', 'Formula', 'Dopant', 'Hc_kOe', 'Tc_K', 'DOI']]

In [ ]:
#checking physical bounds: Ha above Hc, Ha below 2000/Ms kOe, K1 and Tc ceilings
df[(df['Ha_kOe'] <= df['Hc_kOe']) | (df['Ha_kOe'] > 1.05 * 2000 / df['Ms_emu_g']) |
   (df['K_erg_cm3'] > 5.25e6) | (df['K_erg_g'] > 1.0e6) | (df['Tc_K'] > 780)][['Record_ID', 'Formula', 'Ms_emu_g', 'Hc_kOe', 'Ha_kOe', 'K_erg_cm3', 'K_erg_g', 'Tc_K', 'DOI']]

In [ ]:
def parse_formula(formula):
    """
    formula: chemical formula without brackets, e.g. Sr0.7La0.3Fe11.7Co0.3O19
    returns a dictionary {element: index}
    """
    comp = {}
    for el, n in re.findall(r'([A-Z][a-z]?)(\d*\.?\d*)', str(formula)):
        comp[el] = comp.get(el, 0) + (float(n) if n else 1)
    return comp

In [ ]:
#checking that the cation indices of each formula sum to 13
cation_sum = df['Formula'].apply(lambda x: sum(v for k, v in parse_formula(x).items() if k != 'O'))
df[(cation_sum - 13).abs() > 0.05].assign(Cation_sum=cation_sum)[['Record_ID', 'Formula', 'Cation_sum', 'DOI']]

In [ ]:
#checking that the same Ms and Hc do not appear for different compositions
pairs = df.dropna(subset=['Ms_emu_g', 'Hc_kOe'])
dup = pairs.groupby(['Ms_emu_g', 'Hc_kOe']).filter(lambda g: g['Formula'].nunique() > 1)
dup.sort_values(['Ms_emu_g', 'Hc_kOe'])[['Record_ID', 'Formula', 'Ms_emu_g', 'Hc_kOe', 'DOI']]

In [ ]:
atomic_weights = {'Ba': 137.327, 'Sr': 87.62, 'Fe': 55.845, 'O': 15.999, 'La': 138.905, 'Co': 58.933, 'Ti': 47.867,
                  'Zn': 65.38, 'Al': 26.982, 'Ca': 40.078, 'Mn': 54.938, 'Zr': 91.224, 'Ni': 58.693, 'Cr': 51.996,
                  'Cu': 63.546, 'Sm': 150.36, 'Ce': 140.116, 'Nd': 144.242, 'Y': 88.906, 'Gd': 157.25, 'Mg': 24.305,
                  'Sn': 118.71, 'Bi': 208.98, 'Pr': 140.908, 'Tb': 158.925, 'Dy': 162.5, 'Nb': 92.906, 'Si': 28.085,
                  'Ga': 69.723, 'Ru': 101.07, 'V': 50.942, 'In': 114.818, 'Sc': 44.956, 'Cd': 112.414, 'Tm': 168.934,
                  'Eu': 151.964, 'Na': 22.99, 'Pb': 207.2, 'Li': 6.94, 'Er': 167.259, 'Mo': 95.95, 'Ho': 164.93,
                  'W': 183.84, 'Sb': 121.76, 'Hf': 178.49, 'Re': 186.207, 'Yb': 173.045, 'Tl': 204.38, 'Ir': 192.217,
                  'Lu': 174.967, 'K': 39.098, 'Ta': 180.948}
#spin-only moments (in Bohr magnetons) of magnetic ions on Fe sites, other ions are taken as non-magnetic
spin_moments = {'Co': 3, 'Ni': 2, 'Cu': 1, 'Mn': 4, 'Cr': 3, 'V': 2, 'Ru': 2, 'Ir': 1, 'Mo': 2, 'Re': 1}

In [ ]:
def ms_limit(formula, substitution, temperature):
    """
    formula: chemical formula of the sample
    substitution: substituted sites, e.g. Co→Fe;Ti→Fe
    temperature: temperature of the magnetic measurement (K)
    returns the collinear limit of Ms (emu/g): 20 Bohr magnetons per formula unit of the host,
    increased for each ion on an Fe site by 5 minus its spin-only moment, scaled by 0.72 above 250 K
    """
    comp = parse_formula(formula)
    on_fe = {p.split('→')[0] for p in str(substitution).split(';') if '→Fe' in p}
    gain = sum((5 - spin_moments.get(el, 0)) * comp.get(el, 0) for el in on_fe)
    moment = 20 + min(max(gain, 0), 20)
    mass = sum(atomic_weights[el] * n for el, n in comp.items())
    limit = moment * 5585 / mass
    t = pd.to_numeric(temperature, errors='coerce')
    return limit if t < 250 else limit * 0.72

In [ ]:
#checking that Ms does not exceed the collinear limit of the composition by more than 5%
limit = [ms_limit(f, s, t) for f, s, t in zip(df['Formula'], df['Substitution'], df['Measurement_Temperature_K'])]
df[df['Ms_emu_g'] > 1.05 * pd.Series(limit, index=df.index)][['Record_ID', 'Formula', 'Measurement_Temperature_K', 'Ms_emu_g', 'DOI']]

In [ ]:
#checking that all DOIs are valid
dois = df['DOI'].unique().tolist()
for doi in tqdm(dois):
    d = requests.get(f'https://api.crossref.org/works/{doi}')
    if d.status_code != 200:
        print(f'This DOI does not exist: {doi}')